# Inverse Yield Modeling With Gaussian Processes For A Fixed Crop

This notebook mirrors the structure of `inverse_modeling_analysis_MLP.ipynb`, but replaces the forward model with a Gaussian Process (GP) regressor.

The analysis is intentionally restricted to **RED SPRING WHEAT** and does four things:
1. trains a GP model on scaled engineered climate features,
2. quantifies which climate variables matter most using permutation importance,
3. inverts the fitted model to recover plausible climate vectors for low, median, and high yield targets,
4. interprets how the recovered climates differ from the empirical crop-specific climate distribution.

In [19]:
from pathlib import Path
import sys
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF, WhiteKernel
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Manitoba').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MANITOBA_DIR = PROJECT_ROOT / 'Manitoba'
if str(MANITOBA_DIR) not in sys.path:
    sys.path.insert(0, str(MANITOBA_DIR))

from inverse_modeling import inverse_yield, estimate_empirical_reference

plt.style.use('ggplot')
np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore')

DATA_PATH = MANITOBA_DIR / 'prepared' / 'yields_weather_by_crop.csv'
GP_TUNING_PATH = MANITOBA_DIR / 'prepared' / 'gp_hyperparameter_tuning_results.csv'
RANDOM_STATE = 42
CROP = 'RED SPRING WHEAT'
TEST_SIZE = 0.2
TOP_K_IMPORTANT = 12

## 1. Setup And Data Loading

The prepared dataset already contains engineered climate summaries by crop, municipality, and year.

For inverse modeling we hold crop fixed from the start and only optimize over the climate feature vector. The notebook therefore excludes `Year`, `Latitude`, `Longitude`, `Municipality`, and `Crop` from the inversion inputs and keeps only the engineered climate columns matching `^\d{2}_Q[12]_`.

In [20]:
df = pd.read_csv(DATA_PATH)
climate_feature_cols = [c for c in df.columns if re.match(r'^\d{2}_Q[12]_', c)]

fixed_crop_df = (
    df.loc[df['Crop'] == CROP, ['Mean_Yield', 'Crop'] + climate_feature_cols]
      .dropna()
      .reset_index(drop=True)
)

print('Crop:', CROP)
print('Rows after filtering and dropping NA:', len(fixed_crop_df))
print('Number of climate features:', len(climate_feature_cols))
fixed_crop_df.head(3)

Crop: RED SPRING WHEAT
Rows after filtering and dropping NA: 1088
Number of climate features: 72


,Mean_Yield,Crop,05_Q1_CDD,05_Q2_CDD,06_Q1_CDD,06_Q2_CDD,07_Q1_CDD,07_Q2_CDD,08_Q1_CDD,08_Q2_CDD,...,06_Q1_Ptol,06_Q2_Ptol,07_Q1_Ptol,07_Q2_Ptol,08_Q1_Ptol,08_Q2_Ptol,09_Q1_Ptol,09_Q2_Ptol,10_Q1_Ptol,10_Q2_Ptol
0,0.951428,RED SPRING WHEAT,0.0,0.152813,24.440870,34.404783,27.131971,17.745561,28.276071,43.549495,...,0.030000,0.060000,0.030000,0.070000,0.030000,0.030000,0.020000,0.020000,0.000000,0.020000
1,1.203462,RED SPRING WHEAT,0.0,2.287541,35.589788,36.770040,36.366397,17.175362,32.882947,48.528054,...,0.017702,0.036235,0.070022,0.052771,0.060629,0.013699,0.052853,0.013083,0.014716,0.028481
2,1.001053,RED SPRING WHEAT,0.0,0.000000,23.714498,33.048504,26.179830,18.664748,29.415604,46.303922,...,0.032567,0.054740,0.038154,0.078133,0.023669,0.013501,0.020265,0.027254,0.011460,0.028695


## 2. Train GP Model

The MLP notebook standardized the climate inputs before fitting the forward model. We do the same here so inversion and empirical reference estimation happen in the same model-input space.

The GP hyperparameters come from `prepared/gp_hyperparameter_tuning_results.csv`. The best validation row is the one with the lowest validation RMSE. Its optimized kernel is reconstructed explicitly and then used for the final fixed-crop GP fit.

In [21]:
def regression_metrics(y_true, y_pred):
    """Return standard regression metrics for compact train/test reporting."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        'rmse': rmse,
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }


def kernel_from_tuning_row(row):
    """Rebuild a scikit-learn GP kernel from the best tuning-results row."""
    optimized = str(row['optimized_kernel'])

    constant_match = re.search(r'([0-9.]+)\*\*2', optimized)
    length_scale_match = re.search(r'length_scale=([0-9.]+)', optimized)
    noise_match = re.search(r'noise_level=([0-9.]+)', optimized)

    if not (constant_match and length_scale_match and noise_match):
        raise ValueError(f'Could not parse optimized kernel: {optimized}')

    constant_value = float(constant_match.group(1)) ** 2
    length_scale = float(length_scale_match.group(1))
    noise_level = float(noise_match.group(1))

    if row['kernel_name'] == 'rbf':
        base_kernel = RBF(length_scale=length_scale, length_scale_bounds='fixed')
    elif row['kernel_name'] == 'matern_1p5':
        base_kernel = Matern(length_scale=length_scale, nu=1.5, length_scale_bounds='fixed')
    elif row['kernel_name'] == 'matern_2p5':
        base_kernel = Matern(length_scale=length_scale, nu=2.5, length_scale_bounds='fixed')
    else:
        raise ValueError(f"Unsupported kernel_name: {row['kernel_name']}")

    return (
        ConstantKernel(constant_value=constant_value, constant_value_bounds='fixed')
        * base_kernel
        + WhiteKernel(noise_level=noise_level, noise_level_bounds='fixed')
    )


gp_tuning_results = pd.read_csv(GP_TUNING_PATH).sort_values(
    ['val_rmse', 'val_mae', 'val_r2'],
    ascending=[True, True, False],
).reset_index(drop=True)
best_gp_row = gp_tuning_results.iloc[0].copy()
best_gp_row

alpha                                                                   0.01
kernel_name                                                              rbf
val_rmse                                                            0.163162
val_mae                                                             0.122626
val_r2                                                              0.613538
log_marginal_likelihood                                          -656.118604
optimized_kernel           0.833**2 * RBF(length_scale=3.3) + WhiteKernel...
Name: 0, dtype: object

In [22]:
X = fixed_crop_df[climate_feature_cols].copy()
y = fixed_crop_df['Mean_Yield'].to_numpy(dtype=float)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

gp_kernel = kernel_from_tuning_row(best_gp_row)

gp_model = GaussianProcessRegressor(
    kernel=gp_kernel,
    alpha=float(best_gp_row['alpha']),
    normalize_y=True,
    optimizer=None,
    random_state=RANDOM_STATE,
)
gp_model.fit(X_train, y_train)

y_train_pred = gp_model.predict(X_train)
y_test_pred = gp_model.predict(X_test)

metrics = pd.DataFrame([
    {'split': 'train', **regression_metrics(y_train, y_train_pred)},
    {'split': 'test', **regression_metrics(y_test, y_test_pred)},
])
metrics

,split,rmse,mae,r2
0,train,0.118792,0.089364,0.808088
1,test,0.157351,0.122989,0.642838


In [23]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_test_pred, alpha=0.75, edgecolor='white', linewidth=0.5)
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
ax.plot(lims, lims, color='black', linewidth=1)
ax.set_title(f'Observed vs Predicted Yield For {CROP} (Test Set)')
ax.set_xlabel('Observed yield')
ax.set_ylabel('Predicted yield')
plt.tight_layout()
plt.show()

## 3. Feature Importance Analysis

Gaussian Processes do not expose coefficient-style feature effects, so we use permutation importance on the held-out test set. A feature is important when shuffling it produces a clear degradation in predictive performance.

To make the ranking interpretable, the notebook also compares each important feature's empirical distribution with the feature averages observed in the lowest-yield and highest-yield quartiles of the training sample.

In [24]:
permutation = permutation_importance(
    gp_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring='neg_root_mean_squared_error',
)

importance_df = (
    pd.DataFrame({
        'feature': climate_feature_cols,
        'importance_mean': permutation.importances_mean,
        'importance_std': permutation.importances_std,
    })
    .sort_values('importance_mean', ascending=False)
    .reset_index(drop=True)
)
importance_df['importance_rank'] = np.arange(1, len(importance_df) + 1)
top_importance_df = importance_df.head(TOP_K_IMPORTANT).copy()
top_importance_df

,feature,importance_mean,importance_std,importance_rank
0,08_Q1_Ptol,0.012008,0.002402,1
1,05_Q1_CDD,0.009103,0.001830,2
2,05_Q2_CDD,0.007800,0.002229,3
3,05_Q2_Ptol,0.007780,0.002282,4
4,10_Q1_CDD,0.007644,0.001392,5
5,09_Q2_CDD,0.006897,0.001784,6
6,09_Q1_Ptol,0.006842,0.002021,7
7,07_Q2_Ptol,0.006834,0.002494,8
8,06_Q1_Ptol,0.006114,0.001531,9
9,05_Q2_Min_Temp,0.005678,0.001774,10


In [25]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = top_importance_df.sort_values('importance_mean')
ax.barh(plot_df['feature'], plot_df['importance_mean'], xerr=plot_df['importance_std'], color='#2b6cb0')
ax.set_title(f'Top {TOP_K_IMPORTANT} Permutation Importances For GP Yield Prediction')
ax.set_xlabel('Increase in RMSE after permutation')
ax.set_ylabel('Climate feature')
plt.tight_layout()
plt.show()

In [26]:
def parse_feature_name(feature_name):
    """Parse feature names such as 08_Q1_Ptol into year code, quarter, and metric."""
    year_code, quarter, metric = feature_name.split('_', 2)
    return {'year_code': year_code, 'quarter': quarter, 'metric': metric}


def directional_phrase(feature_name, delta):
    """Translate a raw feature shift into a short climate interpretation."""
    parsed = parse_feature_name(feature_name)
    quarter = parsed['quarter']
    metric = parsed['metric']
    if metric == 'Ptol':
        return f"{'more' if delta > 0 else 'less'} {quarter} precipitation"
    if metric == 'HDD':
        return f"{'colder' if delta > 0 else 'warmer'} {quarter}"
    if metric == 'CDD':
        return f"{'more' if delta > 0 else 'less'} {quarter} heat accumulation"
    if metric == 'Max_Temp':
        return f"{'hotter' if delta > 0 else 'cooler'} {quarter} daytime extremes"
    if metric == 'Mean_Temp':
        return f"{'warmer' if delta > 0 else 'cooler'} {quarter}"
    if metric == 'Min_Temp':
        return f"{'warmer' if delta > 0 else 'cooler'} {quarter} nights"
    return f"{'higher' if delta > 0 else 'lower'} {feature_name}"


train_y_series = pd.Series(y_train, index=X_train_raw.index)
low_yield_threshold = float(train_y_series.quantile(0.25))
high_yield_threshold = float(train_y_series.quantile(0.75))

low_yield_mask = train_y_series <= low_yield_threshold
high_yield_mask = train_y_series >= high_yield_threshold

important_feature_rows = []
for row in top_importance_df.itertuples(index=False):
    feature = row.feature
    series = X_train_raw[feature]
    low_mean = float(X_train_raw.loc[low_yield_mask, feature].mean())
    high_mean = float(X_train_raw.loc[high_yield_mask, feature].mean())
    delta_high_minus_low = high_mean - low_mean
    high_yield_direction = (
        'higher values align with higher yield'
        if delta_high_minus_low > 0
        else 'lower values align with higher yield'
    )
    important_feature_rows.append({
        'importance_rank': int(row.importance_rank),
        'feature': feature,
        'importance_mean': float(row.importance_mean),
        'emp_mean': float(series.mean()),
        'emp_std': float(series.std(ddof=1)),
        'emp_min': float(series.min()),
        'q10': float(series.quantile(0.10)),
        'q25': float(series.quantile(0.25)),
        'q50': float(series.quantile(0.50)),
        'q75': float(series.quantile(0.75)),
        'q90': float(series.quantile(0.90)),
        'emp_max': float(series.max()),
        'low_yield_mean': low_mean,
        'high_yield_mean': high_mean,
        'high_minus_low': delta_high_minus_low,
        'high_yield_direction': high_yield_direction,
        'interpretation': directional_phrase(feature, delta_high_minus_low),
    })

important_feature_summary = pd.DataFrame(important_feature_rows)
important_feature_summary

,importance_rank,feature,importance_mean,emp_mean,emp_std,emp_min,q10,q25,q50,q75,q90,emp_max,low_yield_mean,high_yield_mean,high_minus_low,high_yield_direction,interpretation
0,1,08_Q1_Ptol,0.012008,0.029030,0.020353,0.000000,0.006349,0.013294,0.023817,0.040000,0.060000,0.111032,0.029538,0.030342,0.000804,higher values align with higher yield,more Q1 precipitation
1,2,05_Q1_CDD,0.009103,0.441641,0.973711,0.000000,0.000000,0.000000,0.000000,0.000000,2.278377,4.978423,0.624865,0.327892,-0.296973,lower values align with higher yield,less Q1 heat accumulation
2,3,05_Q2_CDD,0.007800,2.726653,4.169138,0.000000,0.000000,0.000000,1.321377,4.157111,6.460453,27.654067,2.486857,3.056600,0.569743,higher values align with higher yield,more Q2 heat accumulation
3,4,05_Q2_Ptol,0.007780,0.044199,0.030948,0.000000,0.010000,0.020000,0.040000,0.054951,0.100000,0.140000,0.043221,0.044748,0.001527,higher values align with higher yield,more Q2 precipitation
4,5,10_Q1_CDD,0.007644,0.924204,2.655777,0.000000,0.000000,0.000000,0.000000,0.000000,2.801976,12.009318,1.517673,0.225654,-1.292019,lower values align with higher yield,less Q1 heat accumulation
5,6,09_Q2_CDD,0.006897,3.412839,4.821461,0.000000,0.000000,0.000000,1.373717,5.133120,10.302609,24.467992,2.116457,5.956107,3.839650,higher values align with higher yield,more Q2 heat accumulation
6,7,09_Q1_Ptol,0.006842,0.020465,0.017692,0.000000,0.000000,0.003155,0.020000,0.030000,0.044244,0.081297,0.016773,0.027109,0.010336,higher values align with higher yield,more Q1 precipitation
7,8,07_Q2_Ptol,0.006834,0.029029,0.024734,0.000000,0.005026,0.010000,0.020000,0.040000,0.070000,0.120000,0.035748,0.023790,-0.011958,lower values align with higher yield,less Q2 precipitation
8,9,06_Q1_Ptol,0.006114,0.045725,0.032340,0.000000,0.010000,0.020000,0.040000,0.070000,0.090079,0.160000,0.048350,0.045606,-0.002744,lower values align with higher yield,less Q1 precipitation
9,10,05_Q2_Min_Temp,0.005678,7.325014,1.844600,2.679547,5.164015,5.662253,7.367967,8.490343,9.882470,12.735861,7.690042,7.113202,-0.576840,lower values align with higher yield,cooler Q2 nights


## 4. Build Inverse Inputs

The inverse solver expects everything in the model's input space, so we estimate the empirical mean and covariance from the **scaled** training matrix.

Bounds are also defined in scaled space using the 1st and 99th percentiles of the training sample. This keeps inverse optimization inside a broad but empirical climate envelope.

In [27]:
reference = estimate_empirical_reference(X_train)
reference_mean = reference['mean']
reference_cov = reference['cov']
reference_cov_inv = np.linalg.pinv(reference_cov)

lower = np.quantile(X_train, 0.01, axis=0)
upper = np.quantile(X_train, 0.99, axis=0)
bounds = list(zip(lower, upper))

train_quantiles = pd.Series(y_train).quantile([0.25, 0.50, 0.75])
targets = {
    'low': float(train_quantiles.loc[0.25]),
    'median': float(train_quantiles.loc[0.50]),
    'high': float(train_quantiles.loc[0.75]),
}

pd.DataFrame({
    'target_label': list(targets.keys()),
    'target_yield': list(targets.values()),
})

,target_label,target_yield
0,low,0.927539
1,median,1.077944
2,high,1.258969


## 5. Run Inverse Modeling

Each target yield is inverted with the same regularization setting used in the MLP notebook's main run:
- `lambda_mahal = 0.0`
- `lambda_l2 = 0.001`
- `n_restarts = 8`

A small helper wraps `inverse_yield()` so optimization failures are captured cleanly instead of breaking the notebook.

In [28]:
def safe_inverse_yield(*, label, y_target, model, feature_names, bounds, reference_mean, reference_cov,
                       lambda_mahal, lambda_l2, n_restarts, random_state):
    """Run inverse optimization and keep failure metadata in a notebook-friendly structure."""
    try:
        result = inverse_yield(
            model=model,
            y_target=y_target,
            feature_names=feature_names,
            bounds=bounds,
            reference_mean=reference_mean,
            reference_cov=reference_cov,
            lambda_mahal=lambda_mahal,
            lambda_l2=lambda_l2,
            n_restarts=n_restarts,
            random_state=random_state,
        )
    except Exception as exc:
        return {
            'scenario': label,
            'x_opt': None,
            'y_target': y_target,
            'y_pred': np.nan,
            'objective_value': np.nan,
            'success': False,
            'message': f'Exception during optimization: {exc}',
            'restart_summaries': [],
        }

    result['scenario'] = label
    return result


inverse_results = {}
for label, y_target in targets.items():
    inverse_results[label] = safe_inverse_yield(
        label=label,
        y_target=y_target,
        model=gp_model,
        feature_names=climate_feature_cols,
        bounds=bounds,
        reference_mean=reference_mean,
        reference_cov=reference_cov,
        lambda_mahal=0.0,
        lambda_l2=0.001,
        n_restarts=8,
        random_state=RANDOM_STATE,
    )

inverse_summary = pd.DataFrame([
    {
        'scenario': label,
        'target_yield': result['y_target'],
        'predicted_yield': result['y_pred'],
        'abs_error': abs(result['y_pred'] - result['y_target']) if np.isfinite(result['y_pred']) else np.nan,
        'objective_value': result['objective_value'],
        'success': result['success'],
        'message': result['message'],
    }
    for label, result in inverse_results.items()
])
inverse_summary

,scenario,target_yield,predicted_yield,abs_error,objective_value,success,message
0,low,0.927539,0.941585,0.014046,0.001187,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
1,median,1.077944,1.072747,0.005197,0.000294,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
2,high,1.258969,1.241038,0.017931,0.004662,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL


## 6. Regularization Sensitivity

Inverse solutions are not unique. For the high-yield target we compare three regularization settings to illustrate the trade-off between target-matching accuracy and climate plausibility.

In [29]:
def mahalanobis_distance(x, mean, inv_cov):
    """Return Mahalanobis distance from the empirical reference mean."""
    delta = x - mean
    return float(np.sqrt(delta.T @ inv_cov @ delta))


nn = NearestNeighbors(n_neighbors=1).fit(X_train)
sensitivity_settings = [
    {'setting': 'none', 'lambda_mahal': 0.0, 'lambda_l2': 0.0},
    {'setting': 'weak_l2', 'lambda_mahal': 0.0, 'lambda_l2': 0.001},
    {'setting': 'mahal_plus_l2', 'lambda_mahal': 0.01, 'lambda_l2': 0.001},
]

sensitivity_rows = []
high_target = targets['high']
for setting in sensitivity_settings:
    result = safe_inverse_yield(
        label='high',
        y_target=high_target,
        model=gp_model,
        feature_names=climate_feature_cols,
        bounds=bounds,
        reference_mean=reference_mean,
        reference_cov=reference_cov,
        lambda_mahal=setting['lambda_mahal'],
        lambda_l2=setting['lambda_l2'],
        n_restarts=8,
        random_state=RANDOM_STATE,
    )

    x_opt = result['x_opt']
    if x_opt is None:
        l2_from_mean = np.nan
        mahal_distance = np.nan
        nearest_train_distance = np.nan
        n_features_at_bounds = np.nan
    else:
        l2_from_mean = float(np.linalg.norm(x_opt - reference_mean))
        mahal_distance = mahalanobis_distance(x_opt, reference_mean, reference_cov_inv)
        nearest_train_distance = float(nn.kneighbors(x_opt.reshape(1, -1), return_distance=True)[0][0, 0])
        n_features_at_bounds = int(np.sum(np.isclose(x_opt, lower, atol=1e-6) | np.isclose(x_opt, upper, atol=1e-6)))

    sensitivity_rows.append({
        'setting': setting['setting'],
        'lambda_mahal': setting['lambda_mahal'],
        'lambda_l2': setting['lambda_l2'],
        'target_yield': result['y_target'],
        'predicted_yield': result['y_pred'],
        'abs_error': abs(result['y_pred'] - result['y_target']) if np.isfinite(result['y_pred']) else np.nan,
        'objective_value': result['objective_value'],
        'success': result['success'],
        'l2_from_reference_mean': l2_from_mean,
        'mahalanobis_distance': mahal_distance,
        'nearest_train_distance': nearest_train_distance,
        'n_features_at_bounds': n_features_at_bounds,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df

,setting,lambda_mahal,lambda_l2,target_yield,predicted_yield,abs_error,objective_value,success,l2_from_reference_mean,mahalanobis_distance,nearest_train_distance,n_features_at_bounds
0,none,0.00,0.000,1.258969,1.258963,0.000005,2.870112e-11,True,6.159252,15.120450,6.171328,1
1,weak_l2,0.00,0.001,1.258969,1.241038,0.017931,4.662227e-03,True,2.083440,3.224535,4.719491,1
2,mahal_plus_l2,0.01,0.001,1.258969,1.055123,0.203846,8.134600e-01,True,5.127885,8.634879,6.922468,1


## 7. Inspect And Interpret Recovered Climate Vectors

The inverse solver returns climate vectors in scaled feature space. We now transform them back to the original climate scale, compare them with the empirical crop-specific mean climate, and identify which features move the most for each yield scenario.

In [30]:
raw_reference_mean = scaler.inverse_transform(reference_mean.reshape(1, -1)).ravel()
raw_reference_mean_series = pd.Series(raw_reference_mean, index=climate_feature_cols)

raw_solutions = {}
for label, result in inverse_results.items():
    if result['x_opt'] is None:
        continue
    raw_solutions[label] = pd.Series(
        scaler.inverse_transform(result['x_opt'].reshape(1, -1)).ravel(),
        index=climate_feature_cols,
    )

comparison = pd.DataFrame({
    'reference_mean': raw_reference_mean_series,
    **{f'{label}_solution': solution for label, solution in raw_solutions.items()},
})
comparison.head()

,reference_mean,low_solution,median_solution,high_solution
05_Q1_CDD,0.441641,0.543907,0.390893,0.261512
05_Q2_CDD,2.726653,3.024279,2.512917,1.716566
06_Q1_CDD,8.186587,8.010541,8.150088,6.738913
06_Q2_CDD,23.020649,23.803648,22.575674,21.225969
07_Q1_CDD,35.134246,36.133999,34.698611,32.182985


In [31]:
def top_feature_shifts(result, top_k=10):
    """Summarize the features with the largest deviation from the empirical mean climate."""
    if result['x_opt'] is None:
        return pd.DataFrame()

    raw_solution = scaler.inverse_transform(result['x_opt'].reshape(1, -1)).ravel()
    delta_scaled = result['x_opt'] - reference_mean
    delta_raw = raw_solution - raw_reference_mean
    out = pd.DataFrame({
        'feature': climate_feature_cols,
        'scaled_shift_from_mean': delta_scaled,
        'abs_scaled_shift': np.abs(delta_scaled),
        'raw_reference_mean': raw_reference_mean,
        'raw_solution': raw_solution,
        'raw_delta': delta_raw,
    })
    out['importance_rank'] = out['feature'].map(importance_df.set_index('feature')['importance_rank'])
    out['importance_mean'] = out['feature'].map(importance_df.set_index('feature')['importance_mean'])
    out['interpretation'] = out.apply(
        lambda row: directional_phrase(row['feature'], row['raw_delta']),
        axis=1,
    )
    return out.sort_values('abs_scaled_shift', ascending=False).head(top_k)


scenario_shift_tables = {}
for label, result in inverse_results.items():
    shift_table = top_feature_shifts(result, top_k=10)
    scenario_shift_tables[label] = shift_table
    print(f'\nScenario: {label}')
    display(shift_table)


Scenario: low


,feature,scaled_shift_from_mean,abs_scaled_shift,raw_reference_mean,raw_solution,raw_delta,importance_rank,importance_mean,interpretation
64,07_Q1_Ptol,0.307027,0.307027,0.045277,0.053623,0.008347,20,0.004319,more Q1 precipitation
70,10_Q1_Ptol,-0.296740,0.296740,0.019658,0.012874,-0.006784,51,0.001933,less Q1 precipitation
65,07_Q2_Ptol,0.268997,0.268997,0.029029,0.035678,0.006649,8,0.006834,more Q2 precipitation
67,08_Q2_Ptol,0.243455,0.243455,0.030505,0.036347,0.005842,14,0.005231,more Q2 precipitation
8,09_Q1_CDD,0.237214,0.237214,12.841302,15.313518,2.472215,12,0.005450,more Q1 heat accumulation
61,05_Q2_Ptol,-0.207904,0.207904,0.044199,0.037769,-0.006431,4,0.007780,less Q2 precipitation
17,07_Q2_HDD,0.199680,0.199680,9.376557,11.058130,1.681573,65,0.000903,colder Q2
7,08_Q2_CDD,-0.194999,0.194999,27.646186,24.361616,-3.284571,45,0.002378,less Q2 heat accumulation
66,08_Q1_Ptol,0.191021,0.191021,0.029030,0.032916,0.003886,1,0.012008,more Q1 precipitation
50,06_Q1_Min_Temp,0.190683,0.190683,10.568101,10.936750,0.368649,47,0.002191,warmer Q1 nights



Scenario: median


,feature,scaled_shift_from_mean,abs_scaled_shift,raw_reference_mean,raw_solution,raw_delta,importance_rank,importance_mean,interpretation
70,10_Q1_Ptol,0.171746,0.171746,0.019658,0.023584,0.003926,51,0.001933,more Q1 precipitation
64,07_Q1_Ptol,-0.149119,0.149119,0.045277,0.041223,-0.004054,20,0.004319,less Q1 precipitation
8,09_Q1_CDD,-0.130920,0.130920,12.841302,11.476872,-1.364430,12,0.005450,less Q1 heat accumulation
17,07_Q2_HDD,-0.125397,0.125397,9.376557,8.320547,-1.056009,65,0.000903,warmer Q2
69,09_Q2_Ptol,0.122160,0.122160,0.019232,0.021303,0.002071,18,0.004579,more Q2 precipitation
67,08_Q2_Ptol,-0.112037,0.112037,0.030505,0.027817,-0.002688,14,0.005231,less Q2 precipitation
61,05_Q2_Ptol,0.106603,0.106603,0.044199,0.047497,0.003297,4,0.007780,more Q2 precipitation
16,07_Q1_HDD,-0.103417,0.103417,11.819264,10.862517,-0.956748,56,0.001446,warmer Q1
7,08_Q2_CDD,0.098467,0.098467,27.646186,29.304768,1.658582,45,0.002378,more Q2 heat accumulation
18,08_Q1_HDD,-0.095361,0.095361,12.235021,11.184077,-1.050944,62,0.001077,warmer Q1



Scenario: high


,feature,scaled_shift_from_mean,abs_scaled_shift,raw_reference_mean,raw_solution,raw_delta,importance_rank,importance_mean,interpretation
70,10_Q1_Ptol,0.636315,0.636315,0.019658,0.034205,0.014547,51,0.001933,more Q1 precipitation
69,09_Q2_Ptol,0.567519,0.567519,0.019232,0.028852,0.009619,18,0.004579,more Q2 precipitation
17,07_Q2_HDD,-0.498233,0.498233,9.376557,5.180767,-4.195790,65,0.000903,warmer Q2
8,09_Q1_CDD,-0.473778,0.473778,12.841302,7.903649,-4.937654,12,0.005450,less Q1 heat accumulation
7,08_Q2_CDD,0.424346,0.424346,27.646186,34.793887,7.147700,45,0.002378,more Q2 heat accumulation
62,06_Q1_Ptol,0.416338,0.416338,0.045725,0.059182,0.013457,9,0.006114,more Q1 precipitation
18,08_Q1_HDD,-0.399661,0.399661,12.235021,7.830478,-4.404543,62,0.001077,warmer Q1
67,08_Q2_Ptol,-0.389962,0.389962,0.030505,0.021148,-0.009357,14,0.005231,less Q2 precipitation
31,08_Q2_Max_Temp,0.375428,0.375428,22.992429,23.895655,0.903226,68,0.000765,hotter Q2 daytime extremes
64,07_Q1_Ptol,-0.359910,0.359910,0.045277,0.035492,-0.009784,20,0.004319,less Q1 precipitation


In [32]:
for label in ['low', 'median', 'high']:
    plot_df = scenario_shift_tables[label].sort_values('scaled_shift_from_mean')
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(plot_df['feature'], plot_df['scaled_shift_from_mean'], color='#2b6cb0')
    ax.axvline(0.0, color='black', linewidth=1)
    ax.set_title(f'Largest Standardized Climate Shifts For The {label.title()} Yield Target')
    ax.set_xlabel('Shift from empirical mean (scaled units)')
    ax.set_ylabel('Climate feature')
    plt.tight_layout()
    plt.show()

In [33]:
heatmap_df = pd.DataFrame({
    label: pd.Series(result['x_opt'] - reference_mean, index=climate_feature_cols)
    for label, result in inverse_results.items()
    if result['x_opt'] is not None
}).T

heatmap_features = (
    heatmap_df.abs().mean(axis=0)
    .sort_values(ascending=False)
    .head(12)
    .index
)
heatmap_plot_df = heatmap_df.loc[:, heatmap_features]

fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(heatmap_plot_df.values, cmap='coolwarm', aspect='auto')
ax.set_xticks(np.arange(len(heatmap_plot_df.columns)))
ax.set_xticklabels(heatmap_plot_df.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(len(heatmap_plot_df.index)))
ax.set_yticklabels(heatmap_plot_df.index)
ax.set_title('Scaled Shifts From Empirical Mean Across Inverse Scenarios')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Scaled shift')
plt.tight_layout()
plt.show()

## 8. Key Findings Summary

The final table focuses on the most policy-relevant comparison: what the model says must move away from the typical climate for **high yield** versus **low yield**, while also keeping the feature-importance ranking visible.

In [34]:
def build_findings_table(scenario_label, top_k=8):
    """Create a compact findings table for one inverse scenario."""
    scenario_df = scenario_shift_tables[scenario_label].copy()
    if scenario_df.empty:
        return scenario_df

    scenario_df['scenario'] = scenario_label
    scenario_df['change_needed'] = np.where(
        scenario_df['raw_delta'] > 0,
        'increase from typical',
        'decrease from typical',
    )
    return scenario_df.loc[:, [
        'scenario',
        'feature',
        'importance_rank',
        'importance_mean',
        'change_needed',
        'scaled_shift_from_mean',
        'raw_delta',
        'raw_reference_mean',
        'raw_solution',
        'interpretation',
    ]].head(top_k)


key_findings_summary = pd.concat(
    [
        build_findings_table('high', top_k=8),
        build_findings_table('low', top_k=8),
    ],
    ignore_index=True,
)
key_findings_summary

,scenario,feature,importance_rank,importance_mean,change_needed,scaled_shift_from_mean,raw_delta,raw_reference_mean,raw_solution,interpretation
0,high,10_Q1_Ptol,51,0.001933,increase from typical,0.636315,0.014547,0.019658,0.034205,more Q1 precipitation
1,high,09_Q2_Ptol,18,0.004579,increase from typical,0.567519,0.009619,0.019232,0.028852,more Q2 precipitation
2,high,07_Q2_HDD,65,0.000903,decrease from typical,-0.498233,-4.195790,9.376557,5.180767,warmer Q2
3,high,09_Q1_CDD,12,0.005450,decrease from typical,-0.473778,-4.937654,12.841302,7.903649,less Q1 heat accumulation
4,high,08_Q2_CDD,45,0.002378,increase from typical,0.424346,7.147700,27.646186,34.793887,more Q2 heat accumulation
5,high,06_Q1_Ptol,9,0.006114,increase from typical,0.416338,0.013457,0.045725,0.059182,more Q1 precipitation
6,high,08_Q1_HDD,62,0.001077,decrease from typical,-0.399661,-4.404543,12.235021,7.830478,warmer Q1
7,high,08_Q2_Ptol,14,0.005231,decrease from typical,-0.389962,-0.009357,0.030505,0.021148,less Q2 precipitation
8,low,07_Q1_Ptol,20,0.004319,increase from typical,0.307027,0.008347,0.045277,0.053623,more Q1 precipitation
9,low,10_Q1_Ptol,51,0.001933,decrease from typical,-0.296740,-0.006784,0.019658,0.012874,less Q1 precipitation


In [35]:
model_quality_summary = metrics.copy()
model_quality_summary['model'] = 'GaussianProcessRegressor'

findings_reference = (
    important_feature_summary.loc[:, [
        'importance_rank',
        'feature',
        'importance_mean',
        'high_yield_direction',
        'interpretation',
        'emp_mean',
        'emp_std',
        'q25',
        'q50',
        'q75',
    ]]
    .sort_values('importance_rank')
    .reset_index(drop=True)
)

print('Model quality summary')
display(model_quality_summary)
print('Inverse optimization summary')
display(inverse_summary)
print('Feature-importance reference')
display(findings_reference)
print('Scenario findings summary')
display(key_findings_summary)

Model quality summary


,split,rmse,mae,r2,model
0,train,0.118792,0.089364,0.808088,GaussianProcessRegressor
1,test,0.157351,0.122989,0.642838,GaussianProcessRegressor


Inverse optimization summary


,scenario,target_yield,predicted_yield,abs_error,objective_value,success,message
0,low,0.927539,0.941585,0.014046,0.001187,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
1,median,1.077944,1.072747,0.005197,0.000294,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
2,high,1.258969,1.241038,0.017931,0.004662,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL


Feature-importance reference


,importance_rank,feature,importance_mean,high_yield_direction,interpretation,emp_mean,emp_std,q25,q50,q75
0,1,08_Q1_Ptol,0.012008,higher values align with higher yield,more Q1 precipitation,0.029030,0.020353,0.013294,0.023817,0.040000
1,2,05_Q1_CDD,0.009103,lower values align with higher yield,less Q1 heat accumulation,0.441641,0.973711,0.000000,0.000000,0.000000
2,3,05_Q2_CDD,0.007800,higher values align with higher yield,more Q2 heat accumulation,2.726653,4.169138,0.000000,1.321377,4.157111
3,4,05_Q2_Ptol,0.007780,higher values align with higher yield,more Q2 precipitation,0.044199,0.030948,0.020000,0.040000,0.054951
4,5,10_Q1_CDD,0.007644,lower values align with higher yield,less Q1 heat accumulation,0.924204,2.655777,0.000000,0.000000,0.000000
5,6,09_Q2_CDD,0.006897,higher values align with higher yield,more Q2 heat accumulation,3.412839,4.821461,0.000000,1.373717,5.133120
6,7,09_Q1_Ptol,0.006842,higher values align with higher yield,more Q1 precipitation,0.020465,0.017692,0.003155,0.020000,0.030000
7,8,07_Q2_Ptol,0.006834,lower values align with higher yield,less Q2 precipitation,0.029029,0.024734,0.010000,0.020000,0.040000
8,9,06_Q1_Ptol,0.006114,lower values align with higher yield,less Q1 precipitation,0.045725,0.032340,0.020000,0.040000,0.070000
9,10,05_Q2_Min_Temp,0.005678,lower values align with higher yield,cooler Q2 nights,7.325014,1.844600,5.662253,7.367967,8.490343


Scenario findings summary


,scenario,feature,importance_rank,importance_mean,change_needed,scaled_shift_from_mean,raw_delta,raw_reference_mean,raw_solution,interpretation
0,high,10_Q1_Ptol,51,0.001933,increase from typical,0.636315,0.014547,0.019658,0.034205,more Q1 precipitation
1,high,09_Q2_Ptol,18,0.004579,increase from typical,0.567519,0.009619,0.019232,0.028852,more Q2 precipitation
2,high,07_Q2_HDD,65,0.000903,decrease from typical,-0.498233,-4.195790,9.376557,5.180767,warmer Q2
3,high,09_Q1_CDD,12,0.005450,decrease from typical,-0.473778,-4.937654,12.841302,7.903649,less Q1 heat accumulation
4,high,08_Q2_CDD,45,0.002378,increase from typical,0.424346,7.147700,27.646186,34.793887,more Q2 heat accumulation
5,high,06_Q1_Ptol,9,0.006114,increase from typical,0.416338,0.013457,0.045725,0.059182,more Q1 precipitation
6,high,08_Q1_HDD,62,0.001077,decrease from typical,-0.399661,-4.404543,12.235021,7.830478,warmer Q1
7,high,08_Q2_Ptol,14,0.005231,decrease from typical,-0.389962,-0.009357,0.030505,0.021148,less Q2 precipitation
8,low,07_Q1_Ptol,20,0.004319,increase from typical,0.307027,0.008347,0.045277,0.053623,more Q1 precipitation
9,low,10_Q1_Ptol,51,0.001933,decrease from typical,-0.296740,-0.006784,0.019658,0.012874,less Q1 precipitation


In [36]:

# Résumé des résultats clés pour analyse
print('=== Metrics ===')
print(metrics.to_string(index=False))

print('\n=== Top 12 features par importance de permutation ===')
print(top_importance_df.head(12).to_string(index=False))

print('\n=== Résumé inverse optimisation ===')
print(inverse_summary.to_string(index=False))

print('\n=== Résumé des principales caractéristiques des scénarios inverse ===')
print(key_findings_summary.to_string(index=False))


=== Metrics ===
split     rmse      mae       r2
train 0.118792 0.089364 0.808088
 test 0.157351 0.122989 0.642838

=== Top 12 features par importance de permutation ===
       feature  importance_mean  importance_std  importance_rank
    08_Q1_Ptol         0.012008        0.002402                1
     05_Q1_CDD         0.009103        0.001830                2
     05_Q2_CDD         0.007800        0.002229                3
    05_Q2_Ptol         0.007780        0.002282                4
     10_Q1_CDD         0.007644        0.001392                5
     09_Q2_CDD         0.006897        0.001784                6
    09_Q1_Ptol         0.006842        0.002021                7
    07_Q2_Ptol         0.006834        0.002494                8
    06_Q1_Ptol         0.006114        0.001531                9
05_Q2_Min_Temp         0.005678        0.001774               10
    10_Q2_Ptol         0.005593        0.001373               11
     09_Q1_CDD         0.005450        0.001681   

In [37]:

# Résumé compact des résultats essentiels
print('=== Metrics (compacts) ===')
print(metrics.to_dict('records'))

print('\n=== Top 8 features ===')
for row in top_importance_df.head(8).itertuples(index=False):
    print(f"{row.feature}: importance_mean={row.importance_mean:.6f}, importance_std={row.importance_std:.6f}")

print('\n=== Inverse optimisation summary ===')
for row in inverse_summary.itertuples(index=False):
    print(f"{row.scenario}: target={row.target_yield:.4f}, pred={row.predicted_yield:.4f}, abs_error={row.abs_error:.4f}, success={row.success}")

print('\n=== Scénarios clés pour high/low ===')
for row in key_findings_summary.itertuples(index=False):
    print(f"{row.scenario} | {row.feature} | rank={int(row.importance_rank)} | change={row.change_needed} | raw_delta={row.raw_delta:.4f} | interp={row.interpretation}")


=== Metrics (compacts) ===
[{'split': 'train', 'rmse': 0.11879183293146278, 'mae': 0.08936434744796506, 'r2': 0.8080883144910325}, {'split': 'test', 'rmse': 0.15735108395250658, 'mae': 0.12298862484649416, 'r2': 0.6428383292441653}]

=== Top 8 features ===
08_Q1_Ptol: importance_mean=0.012008, importance_std=0.002402
05_Q1_CDD: importance_mean=0.009103, importance_std=0.001830
05_Q2_CDD: importance_mean=0.007800, importance_std=0.002229
05_Q2_Ptol: importance_mean=0.007780, importance_std=0.002282
10_Q1_CDD: importance_mean=0.007644, importance_std=0.001392
09_Q2_CDD: importance_mean=0.006897, importance_std=0.001784
09_Q1_Ptol: importance_mean=0.006842, importance_std=0.002021
07_Q2_Ptol: importance_mean=0.006834, importance_std=0.002494

=== Inverse optimisation summary ===
low: target=0.9275, pred=0.9416, abs_error=0.0140, success=True
median: target=1.0779, pred=1.0727, abs_error=0.0052, success=True
high: target=1.2590, pred=1.2410, abs_error=0.0179, success=True

=== Scénarios cl